
# Notebook 12 · Propensión de compra por categoría

## Pregunta de negocio
¿Qué categoría debería recomendarse a cada cliente en la siguiente activación?

## Qué aprenderás
- Cómo combinar navegación web y compras históricas.
- Cómo entrenar un modelo multiclase orientado a recomendación.
- Cómo interpretar un resultado de top-3 categorías.

## Outputs
- `nb12_propension_categoria_scoring.csv`
- `nb12_propension_categoria_training.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

fact = tables["fact"].copy()
eventos_web = tables["eventos_web"].copy()

sales = fact[(fact["tipo_movimiento"] == "VENTA") & fact["customer_id"].notna()].copy()
sales["categoria_norm"] = sales["categoria"].astype(str).str.strip().str.title()
eventos_web["categoria_norm"] = eventos_web["categoria"].astype(str).str.strip().str.title()

snapshots = pd.date_range("2024-06-01", "2025-08-01", freq="2MS")
rows = []

for cust in sales["customer_id"].dropna().unique():
    cust_sales = sales[sales["customer_id"] == cust].sort_values("fecha_movimiento_dt")
    cust_web = eventos_web[eventos_web["customer_id"] == cust].sort_values("fecha_hora_evento_dt")

    for snap in snapshots:
        hist_sales = cust_sales[(cust_sales["fecha_movimiento_dt"] < snap) & (cust_sales["fecha_movimiento_dt"] >= snap - pd.Timedelta(days=120))]
        hist_web = cust_web[(cust_web["fecha_hora_evento_dt"] < snap) & (cust_web["fecha_hora_evento_dt"] >= snap - pd.Timedelta(days=30))]
        future = cust_sales[(cust_sales["fecha_movimiento_dt"] >= snap) & (cust_sales["fecha_movimiento_dt"] < snap + pd.Timedelta(days=60))]
        target_counts = future.groupby("categoria_norm")["importe_abs_num"].sum().sort_values(ascending=False)

        if (hist_sales.empty and hist_web.empty) or target_counts.empty:
            continue

        target_category = target_counts.index[0]
        if str(target_category).lower() == "nan":
            continue

        rows.append({
            "customer_id": cust,
            "snapshot_date": snap,
            "web_pageviews_30d": int((hist_web["tipo_evento"] == "page_view").sum()),
            "web_searches_30d": int((hist_web["tipo_evento"] == "search").sum()),
            "web_add_to_cart_30d": int((hist_web["tipo_evento"] == "add_to_cart").sum()),
            "web_checkout_30d": int((hist_web["tipo_evento"] == "checkout_start").sum()),
            "hist_tx_120d": hist_sales["doc_id_std"].nunique(),
            "hist_revenue_120d": hist_sales["importe_abs_num"].sum(),
            "hist_discount_120d": hist_sales["descuento_pct_num"].fillna(0).mean(),
            "hist_category_diversity_120d": hist_sales["categoria_norm"].nunique(),
            "last_category": hist_sales.sort_values("fecha_movimiento_dt")["categoria_norm"].iloc[-1] if len(hist_sales) else np.nan,
            "top_viewed_category": hist_web["categoria_norm"].dropna().mode().iloc[0] if hist_web["categoria_norm"].notna().any() else np.nan,
            "target_category": target_category,
        })

prop_train = pd.DataFrame(rows)
prop_train.shape, prop_train["target_category"].value_counts()


((146, 13),
 target_category
 Muebles         51
 Decoración      42
 Iluminación     32
 Textil Hogar    21
 Name: count, dtype: int64)

In [3]:

X = prop_train.drop(columns=["customer_id", "snapshot_date", "target_category"])
y = prop_train["target_category"]

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

model = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
model.fit(X_train, y_train)

proba_test = model.predict_proba(X_test)
classes = model.named_steps["clf"].classes_
pred_test = model.predict(X_test)

metrics = pd.DataFrame({
    "metric": ["accuracy_top1", "accuracy_top3"],
    "value": [
        accuracy_score(y_test, pred_test),
        top_k_accuracy_score(y_test, proba_test, k=min(3, len(classes)), labels=classes)
    ]
})
metrics


,metric,value
0,accuracy_top1,0.318182
1,accuracy_top3,0.818182


In [4]:

latest_snap = prop_train["snapshot_date"].max()
latest_scoring = prop_train[prop_train["snapshot_date"] == latest_snap].copy()

score_matrix = model.predict_proba(latest_scoring.drop(columns=["customer_id", "snapshot_date", "target_category"]))
score_df = pd.DataFrame(score_matrix, columns=model.named_steps["clf"].classes_, index=latest_scoring.index)

top3 = []
for idx, row in score_df.iterrows():
    ordered = row.sort_values(ascending=False).head(3)
    rec = latest_scoring.loc[idx, ["customer_id", "last_category", "top_viewed_category"]].to_dict()
    for rank, (cat, score) in enumerate(ordered.items(), start=1):
        rec[f"categoria_rank_{rank}"] = cat
        rec[f"score_rank_{rank}"] = score
    top3.append(rec)

scoring = pd.DataFrame(top3)
scoring["recomendacion_campana"] = np.where(
    scoring["categoria_rank_1"].eq(scoring["top_viewed_category"]),
    "Activar mensaje de conversión con categoría ya explorada",
    "Activar descubrimiento guiado con prueba social"
)
scoring.head(15)


,customer_id,last_category,top_viewed_category,categoria_rank_1,score_rank_1,categoria_rank_2,score_rank_2,categoria_rank_3,score_rank_3,recomendacion_campana
0,C0006,Muebles,NaN,Muebles,0.790000,Decoración,0.123333,Iluminación,0.046667,Activar descubrimiento guiado con prueba social
1,C0075,Textil Hogar,NaN,Muebles,0.750000,Decoración,0.170000,Textil Hogar,0.056667,Activar descubrimiento guiado con prueba social
2,C0033,NaN,Muebles,Decoración,0.511771,Muebles,0.284044,Iluminación,0.204185,Activar descubrimiento guiado con prueba social
3,C0064,Textil Hogar,NaN,Decoración,0.413333,Muebles,0.400000,Iluminación,0.116667,Activar descubrimiento guiado con prueba social
4,C0001,Textil Hogar,NaN,Decoración,0.820000,Muebles,0.113333,Textil Hogar,0.050000,Activar descubrimiento guiado con prueba social
5,C0081,Iluminación,Muebles,Decoración,0.686667,Iluminación,0.169131,Muebles,0.124203,Activar descubrimiento guiado con prueba social
6,C0009,Iluminación,NaN,Decoración,0.636667,Muebles,0.236667,Iluminación,0.093333,Activar descubrimiento guiado con prueba social
7,C0044,NaN,Muebles,Muebles,0.579707,Iluminación,0.407956,Decoración,0.012337,Activar mensaje de conversión con categoría ya...
8,C0011,Decoración,NaN,Iluminación,0.637460,Decoración,0.229896,Muebles,0.099311,Activar descubrimiento guiado con prueba social
9,C0089,NaN,Iluminación,Textil Hogar,0.582538,Muebles,0.371697,Decoración,0.037697,Activar descubrimiento guiado con prueba social



## Lectura de negocio

Para activación de marketing no hace falta acertar siempre la categoría exacta en primera posición. Lo relevante es disponer de:

- un top-3 razonable,
- una categoría dominante para priorizar creatividad,
- y una explicación simple a partir de navegación y compra histórica.


In [5]:

path_train = mpu.export_csv(prop_train, "nb12_propension_categoria_training.csv")
path_score = mpu.export_csv(scoring.sort_values("score_rank_1", ascending=False), "nb12_propension_categoria_scoring.csv")

print("Guardado:", path_train)
print("Guardado:", path_score)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb12_propension_categoria_training.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb12_propension_categoria_scoring.csv
